# Ball detection vs inference resolution

Diagnostic spike. Question: is weak ball detection mostly an artifact of YOLO
resizing 1920x1080 frames down to 640 before inference?

A ball about 15 px across at full resolution becomes about 5 px at 640, which is
close to invisible. Players survive the shrink, the ball may not.

Measured per resolution: share of frames with a ball detection, and latency split by
stage. Since 2026-09-25 the ball skips ByteTrack and comes straight from the
detector, and `max_ball_miss=0` disables the carry-forward, so the ball count is the
detector's own.

Tuning range only: frames 0 to 299. Frames 300 to 749 are held back for the
hand-label spot-check (see `decision_log.md`, 2026-09-25).

No ground truth here, so this counts detections, not correct detections. A higher
count could include false positives. The hand-label spot-check is what settles
precision.

In [1]:
import sys
sys.path.append("..")

import cv2
import numpy as np
from pipeline.detection.tracker import Tracker

%load_ext autoreload
%autoreload 2

WEIGHTS = "../pipeline/detection/football_yolo26n_best.pt"
SRC = r"C:\Users\rohan\Desktop\Quant Sports Project\Tester video\08fd33_4.mp4"
N_FRAMES = 300  # tuning range: frames 0 to 299 only

C:\Users\rohan\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def measure(imgsz, n_frames=N_FRAMES, conf=0.1):
    """Run n_frames at one inference resolution. Returns ball coverage and latency."""
    tracker = Tracker(WEIGHTS, conf=conf, imgsz=imgsz, max_ball_miss=0)

    cap = cv2.VideoCapture(SRC)
    assert cap.isOpened(), f"Failed to open {SRC}"

    ball_seen = []          # one True/False per frame, reused for the gap analysis
    multi_ball_frames = 0
    stage_ms = []

    for i in range(n_frames):
        ok, frame = cap.read()
        if not ok:
            break

        tracks, raw = tracker.track_frame(frame)
        if i > 0:  # frame 0 includes model warmup
            stage_ms.append(tracker.last_timings)

        ball_seen.append(bool((tracks.class_id == tracker.ball_class_id).any()))
        # The tracker keeps one ball per frame, so count extra balls in raw output.
        raw_cls = raw.boxes.cls.cpu().numpy()
        raw_conf = raw.boxes.conf.cpu().numpy()
        n_raw_ball = int(((raw_cls == tracker.ball_class_id) & (raw_conf >= tracker.ball_conf)).sum())
        if n_raw_ball > 1:
            multi_ball_frames += 1

    cap.release()

    total = [sum(t.values()) for t in stage_ms]
    return {
        "imgsz": imgsz,
        "frames": len(ball_seen),
        "ball_seen": ball_seen,
        "ball_pct": 100 * sum(ball_seen) / len(ball_seen),
        "multi_ball": multi_ball_frames,
        "mean_ms": float(np.mean(total)),
        "p95_ms": float(np.percentile(total, 95)),
        "stages": {k: float(np.mean([t[k] for t in stage_ms])) for k in stage_ms[0]},
    }

In [3]:
results = [measure(s) for s in (640, 960, 1280)]

print(f"{'imgsz':>6} {'ball %':>8} {'>1 ball':>8} {'mean ms':>9} {'p95 ms':>8}  {'25fps?':>7}")
for r in results:
    fits = "yes" if r["mean_ms"] <= 40 else "no"
    print(f"{r['imgsz']:>6} {r['ball_pct']:>7.1f}% {r['multi_ball']:>8} "
          f"{r['mean_ms']:>9.1f} {r['p95_ms']:>8.1f}  {fits:>7}")

print("\nMean ms per stage:")
for r in results:
    parts = "  ".join(f"{k} {v:.2f}" for k, v in r["stages"].items())
    print(f"{r['imgsz']:>6}: {parts}")

C:\Users\rohan\Desktop\Quant Sports Project\experiments\..\pipeline\detection\tracker.py:26: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  self.tracker = sv.ByteTrack()


 imgsz   ball %  >1 ball   mean ms   p95 ms   25fps?
   640    87.0%      157      42.9     51.2       no
   960    92.3%      145      62.5     66.6       no
  1280    92.0%      112     100.9    109.2       no

Mean ms per stage:
   640: yolo 35.94  split_nms 0.72  bytetrack 6.12  ball 0.09
   960: yolo 55.68  split_nms 0.74  bytetrack 6.01  ball 0.09
  1280: yolo 93.97  split_nms 0.72  bytetrack 6.09  ball 0.09


In [4]:
# Runs of consecutive missed frames at each resolution.
# The longest run sets how long a gap the Kalman filter would have to bridge.
for r in results:
    gaps = []   # (start frame, length)
    run_start = None
    for i, seen in enumerate(r["ball_seen"]):
        if not seen and run_start is None:
            run_start = i
        elif seen and run_start is not None:
            gaps.append((run_start, i - run_start))
            run_start = None
    if run_start is not None:
        gaps.append((run_start, len(r["ball_seen"]) - run_start))

    lengths = [n for _, n in gaps]
    longest = max(gaps, key=lambda g: g[1]) if gaps else (None, 0)
    print(f"imgsz {r['imgsz']}: {len(gaps)} gaps, longest {longest[1]} frames "
          f"({longest[1] / 25:.2f} s, starting frame {longest[0]}), "
          f"median {int(np.median(lengths)) if lengths else 0} frames")

imgsz 640: 8 gaps, longest 9 frames (0.36 s, starting frame 170), median 5 frames
imgsz 960: 7 gaps, longest 9 frames (0.36 s, starting frame 117), median 2 frames
imgsz 1280: 9 gaps, longest 6 frames (0.24 s, starting frame 120), median 2 frames


## Findings (2026-09-25, frames 0 to 299, CPU, conf and ball_conf 0.1)

| imgsz | Frames with ball | Mean ms | P95 ms | YOLO ms | ByteTrack ms |
|---|---|---|---|---|---|
| 640 | 87.0% | 42.9 | 51.2 | 35.9 | 6.1 |
| 960 | 92.3% | 62.5 | 66.6 | 55.7 | 6.0 |
| 1280 | 92.0% | 100.9 | 109.2 | 94.0 | 6.1 |

- **Resolution is not the main cause of weak ball detection.** Going from 640 to
  960 adds about 5 points of ball coverage for about 20 ms more per frame. 1280 adds
  nothing over 960. **Decision: keep 640.**
- **YOLO is about 84% of the latency.** Split plus NMS is about 0.7 ms, ByteTrack
  about 6 ms, ball handling about 0.1 ms. The 2026-09-22 estimate (NMS plus
  ByteTrack about 13 ms) was too high. At 640 the pipeline is 42.9 ms mean on CPU,
  just over the 40 ms budget; any speed-up has to come from the detector or hardware.
- **">1 ball" is mostly duplicate boxes on the same ball.** In the 157 frames at 640
  with two or more ball detections, the top two are a median 3 px apart (the ball
  now skips NMS). About 10% are over 700 px apart, which are real false balls.
  Picking the most confident box handles the duplicates; the Kalman gate should
  handle the distant ones.
- Top ball confidence: median 0.51; 24 of 261 frames have a top ball under 0.25.
- **Gaps at 640:** 8 gaps, longest 9 frames (0.36 s, starting frame 170), median
  5. So the Kalman filter needs to bridge about 10 frames; `max_ball_miss` around
  10 is the starting point.

Counts, not correct detections: precision is still unknown until the hand-label
spot-check.